In [38]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, roc_auc_score
import torch.nn.functional as F


In [39]:
# Treningowy — z AutoAugment (CIFAR-10 policy)
transform_train = transforms.Compose([
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Testowy — tylko normalizacja, bez augmentacji
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

trainset = torchvision.datasets.CIFAR10(root="./data", train=True,  download=False, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root="./data", train=False, download=False, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True,  num_workers=0)
testloader  = DataLoader(testset,  batch_size=64, shuffle=False, num_workers=0)

print(f"Train: {len(trainset)} | Test: {len(testset)}")

Train: 50000 | Test: 10000


In [40]:
device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()          else "cpu"
)
print(f"Device: {device}")


Device: mps


In [41]:
# Obliczamy mean i std na surowych pikselach [0, 1]
_tmp = DataLoader(
    torchvision.datasets.CIFAR10(root="./data", train=True,
                                 download=True, transform=transforms.ToTensor()),
    batch_size=512
)
_mean, _std, _n = torch.zeros(3), torch.zeros(3), 0
for imgs, _ in _tmp:
    _mean += imgs.mean(dim=[0, 2, 3])
    _std  += imgs.std(dim=[0, 2, 3])
    _n    += 1
MEAN = (_mean / _n).tolist()
STD  = (_std  / _n).tolist()
print(f'MEAN: {[f"{v:.4f}" for v in MEAN]}')
print(f'STD:  {[f"{v:.4f}" for v in STD]}')


MEAN: ['0.4914', '0.4822', '0.4466']
STD:  ['0.2470', '0.2434', '0.2615']


In [42]:
# Treningowy — z AutoAugment (CIFAR-10 policy)
transform_train = transforms.Compose([
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Testowy — tylko normalizacja, bez augmentacji
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

trainset = torchvision.datasets.CIFAR10(root="./data", train=True,  download=False, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root="./data", train=False, download=False, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True,  num_workers=0)
testloader  = DataLoader(testset,  batch_size=64, shuffle=False, num_workers=0)

print(f"Train: {len(trainset)} | Test: {len(testset)}")

Train: 50000 | Test: 10000


### CNN better

In [43]:
class CnnBetter(nn.Module):
    def __init__(self, num_classes: int = 10, dropout_p: float = 0.3):
        super().__init__()
        def block(in_c, out_c, drop):
            # Podwajamy kanały gdy zmniejszamy mapę o połowę — zachowujemy
            # całkowitą "pojemność informacyjną" warstwy (in_c * H * W ≈ out_c * H/2 * W/2)
            return [
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
                nn.MaxPool2d(2), nn.Dropout(drop),
            ]
        self.layers = nn.Sequential(
            *block(3,  32,  dropout_p),   # kanały: 3→32,   mapa: 32×32→16×16
            *block(32, 64,  dropout_p),   # kanały: 32→64,  mapa: 16×16→8×8
            *block(64, 128, dropout_p),   # kanały: 64→128, mapa: 8×8→4×4
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):         return self.layers(x)
    def predict_proba(self, x):   return torch.softmax(self(x), dim=1)
    def predict(self, x):         return torch.argmax(self.predict_proba(x), dim=1)

### MLP

In [44]:
class MLP(nn.Module):
    def __init__(self, input_size: int, num_classes: int = 10, dropout_p: float = 0.2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(512, 256),        nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):         return self.layers(x)
    def predict_proba(self, x):   return torch.softmax(self(x), dim=1)
    def predict(self, x):         return torch.argmax(self.predict_proba(x), dim=1)


### CNN (mvp)

In [45]:
class CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.layers = nn.Sequential(
            # blok 1
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),
            # blok 2
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),
            # klasyfikator
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):         return self.layers(x)
    def predict_proba(self, x):   return torch.softmax(self(x), dim=1)
    def predict(self, x):         return torch.argmax(self.predict_proba(x), dim=1)


In [46]:
MODEL_REGISTRY = {
    "MLP":        (MLP,       {"input_size": 3 * 32 * 32}, "MLP_cifar10.pth"),
    "CNN":        (CNN,       {},                           "CNN_cifar10.pth"),
    "CNN_BETTER": (CnnBetter, {},                           "CNN_BETTER_cifar10.pth"),
    "CNN_ROBUST": (CnnBetter, {},                           "CNN_ROBUST_cifar10.pth"),
    # "CNN_ACC":    (CnnAcc,    {},                           "CNN_ACC_cifar10.pth"),
}

def load_model(name: str) -> nn.Module:
    """Wczytuje model z pliku .pth na podstawie MODEL_REGISTRY."""
    ModelClass, kwargs, pth = MODEL_REGISTRY[name]
    m = ModelClass(**kwargs).to(device)
    m.load_state_dict(torch.load(pth, map_location=device, weights_only=True))
    m.eval()
    return m


In [47]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, pred, target):
        """
        pred: model predictions (logits), shape [batch_size, num_classes]
        target: ground truth labels (class indices), shape [batch_size]
        """
        num_classes = pred.size(-1)
        log_probs = F.log_softmax(pred, dim=-1)

        # Create smoothed target distribution
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)

        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))

In [48]:
def train_with_label_smoothing(model, train_loader, epochs=50, smoothing=0.1, lr=0.001):
    criterion = LabelSmoothingCrossEntropy(smoothing=smoothing)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        acc = 100. * correct / total
        print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Acc: {acc:.2f}%')

    return model

In [49]:
def test_robustness(model, test_loader, attack_fn, attack_name):
    """
    Tests model robustness against adversarial attacks.

    attack_fn: function that takes (model, images, labels) and returns adversarial examples
    """
    model.eval()
    correct_clean = 0
    correct_adv = 0
    total = 0

    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Clean accuracy
        with torch.no_grad():
            outputs_clean = model(images)
            _, pred_clean = outputs_clean.max(1)
            correct_clean += pred_clean.eq(labels).sum().item()

        # Adversarial accuracy
        adv_images = attack_fn(model, images, labels)
        with torch.no_grad():
            outputs_adv = model(adv_images)
            _, pred_adv = outputs_adv.max(1)
            correct_adv += pred_adv.eq(labels).sum().item()

        total += labels.size(0)

    clean_acc = 100. * correct_clean / total
    adv_acc = 100. * correct_adv / total

    print(f"\n{attack_name} Results:")
    print(f"Clean Accuracy: {clean_acc:.2f}%")
    print(f"Adversarial Accuracy: {adv_acc:.2f}%")
    print(f"Robustness (drop): {clean_acc - adv_acc:.2f}%")

    return clean_acc, adv_acc

In [50]:
def fgsm_attack(model, images, labels, epsilon=0.03):
    """Fast Gradient Sign Method"""
    images.requires_grad = True
    outputs = model(images)
    loss = F.cross_entropy(outputs, labels)
    model.zero_grad()
    loss.backward()

    adv_images = images + epsilon * images.grad.sign()
    adv_images = torch.clamp(adv_images, 0, 1)
    return adv_images.detach()

def pgd_attack(model, images, labels, epsilon=0.03, alpha=0.01, iters=10):
    """Projected Gradient Descent"""
    adv_images = images.clone().detach()

    for _ in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = F.cross_entropy(outputs, labels)
        model.zero_grad()
        loss.backward()

        adv_images = adv_images + alpha * adv_images.grad.sign()
        eta = torch.clamp(adv_images - images, -epsilon, epsilon)
        adv_images = torch.clamp(images + eta, 0, 1).detach()

    return adv_images

### Testing

In [52]:
# 1. Load dataset
# train_loader, test_loader = load_model("CNN")
train_loader, test_loader = trainloader,testloader

# 2. Train baseline model (no label smoothing)
model_baseline = CNN().to(device)
model_baseline = train_with_label_smoothing(model_baseline, train_loader, epochs=12, smoothing=0.0)
torch.save(model_baseline.state_dict(), "CNN_baseline_cifar10.pth")

# 3. Train with label smoothing
model_ls = CNN().to(device)
model_ls = train_with_label_smoothing(model_ls, train_loader, epochs=12, smoothing=0.1)
torch.save(model_ls.state_dict(), "CNN_label_smoothing_cifar10.pth")

# 4. Compare robustness
print("\n=== BASELINE MODEL ===")
test_robustness(model_baseline, test_loader,
                lambda m, x, y: fgsm_attack(m, x, y, epsilon=0.03), "FGSM ε=0.03")
test_robustness(model_baseline, test_loader,
                lambda m, x, y: pgd_attack(m, x, y, epsilon=0.03), "PGD ε=0.03")

print("\n=== LABEL SMOOTHING MODEL ===")
test_robustness(model_ls, test_loader,
                lambda m, x, y: fgsm_attack(m, x, y, epsilon=0.03), "FGSM ε=0.03")
test_robustness(model_ls, test_loader,
                lambda m, x, y: pgd_attack(m, x, y, epsilon=0.03), "PGD ε=0.03")

Epoch 1/12, Loss: 1.7197, Acc: 36.90%
Epoch 2/12, Loss: 1.3426, Acc: 52.28%
Epoch 3/12, Loss: 1.2028, Acc: 57.36%
Epoch 4/12, Loss: 1.1330, Acc: 60.12%
Epoch 5/12, Loss: 1.0806, Acc: 61.99%
Epoch 6/12, Loss: 1.0389, Acc: 63.46%
Epoch 7/12, Loss: 1.0167, Acc: 64.40%
Epoch 8/12, Loss: 0.9878, Acc: 65.36%
Epoch 9/12, Loss: 0.9748, Acc: 65.91%
Epoch 10/12, Loss: 0.9651, Acc: 66.24%
Epoch 11/12, Loss: 0.9316, Acc: 67.41%
Epoch 12/12, Loss: 0.9313, Acc: 67.53%
Epoch 1/12, Loss: 1.8597, Acc: 37.44%
Epoch 2/12, Loss: 1.5736, Acc: 52.79%
Epoch 3/12, Loss: 1.4765, Acc: 58.15%
Epoch 4/12, Loss: 1.4293, Acc: 60.79%
Epoch 5/12, Loss: 1.3873, Acc: 62.79%
Epoch 6/12, Loss: 1.3672, Acc: 63.88%
Epoch 7/12, Loss: 1.3466, Acc: 65.10%
Epoch 8/12, Loss: 1.3289, Acc: 66.15%
Epoch 9/12, Loss: 1.3174, Acc: 66.41%
Epoch 10/12, Loss: 1.3053, Acc: 67.15%
Epoch 11/12, Loss: 1.2964, Acc: 67.71%
Epoch 12/12, Loss: 1.2840, Acc: 68.47%

=== BASELINE MODEL ===

FGSM ε=0.03 Results:
Clean Accuracy: 77.72%
Adversarial A

(77.86, 17.06)

### Train with gradient clipping

In [53]:
def train_with_gradient_clipping(model, train_loader, epochs=50, lr=0.001, clip_value=1.0):
    """
    Train model with gradient clipping.

    clip_value: maximum gradient norm (typical values: 0.5, 1.0, 2.0)
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            # Apply gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)

            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        acc = 100. * correct / total
        print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Acc: {acc:.2f}%')

    return model

In [55]:
# 1. Train baseline model (no gradient clipping)
# model_baseline = CNN().to(device)
# model_baseline = train_with_gradient_clipping(model_baseline, train_loader,
#                                               epochs=50, clip_value=float('inf'))  # No clipping
# torch.save(model_baseline.state_dict(), "CNN_baseline_cifar10.pth")

# 2. Train with gradient clipping
model_gc = CNN().to(device)
model_gc = train_with_gradient_clipping(model_gc, train_loader,
                                        epochs=12, clip_value=1.0)  # Clip at norm=1.0
torch.save(model_gc.state_dict(), "CNN_gradient_clipping_cifar10.pth")

# 3. Compare robustness
print("\n=== BASELINE MODEL ===")
test_robustness(model_baseline, test_loader,
                lambda m, x, y: fgsm_attack(m, x, y, epsilon=0.03), "FGSM ε=0.03")
test_robustness(model_baseline, test_loader,
                lambda m, x, y: pgd_attack(m, x, y, epsilon=0.03), "PGD ε=0.03")

print("\n=== GRADIENT CLIPPING MODEL ===")
test_robustness(model_gc, test_loader,
                lambda m, x, y: fgsm_attack(m, x, y, epsilon=0.03), "FGSM ε=0.03")
test_robustness(model_gc, test_loader,
                lambda m, x, y: pgd_attack(m, x, y, epsilon=0.03), "PGD ε=0.03")

Epoch 1/12, Loss: 1.7329, Acc: 36.64%
Epoch 2/12, Loss: 1.3755, Acc: 50.72%
Epoch 3/12, Loss: 1.2277, Acc: 56.45%
Epoch 4/12, Loss: 1.1472, Acc: 59.78%
Epoch 5/12, Loss: 1.0822, Acc: 61.82%
Epoch 6/12, Loss: 1.0455, Acc: 63.24%
Epoch 7/12, Loss: 1.0110, Acc: 64.38%
Epoch 8/12, Loss: 0.9766, Acc: 65.70%
Epoch 9/12, Loss: 0.9638, Acc: 66.30%
Epoch 10/12, Loss: 0.9526, Acc: 66.68%
Epoch 11/12, Loss: 0.9352, Acc: 67.30%
Epoch 12/12, Loss: 0.9197, Acc: 68.04%

=== BASELINE MODEL ===

FGSM ε=0.03 Results:
Clean Accuracy: 73.50%
Adversarial Accuracy: 33.53%
Robustness (drop): 39.97%

PGD ε=0.03 Results:
Clean Accuracy: 73.50%
Adversarial Accuracy: 15.56%
Robustness (drop): 57.94%

=== GRADIENT CLIPPING MODEL ===

FGSM ε=0.03 Results:
Clean Accuracy: 77.24%
Adversarial Accuracy: 38.09%
Robustness (drop): 39.15%

PGD ε=0.03 Results:
Clean Accuracy: 77.24%
Adversarial Accuracy: 16.95%
Robustness (drop): 60.29%


(77.24, 16.95)